In [ ]:
using Base.Threads
println( "Number of threads: ", nthreads() )

include( "../args.jl" )
include( "../model.jl" )
include( "../geom.jl" )
include( "../recur.jl" )

loadsteps = true
savesteps = !loadsteps
loadsteps, savesteps

In [ ]:
# Case and data folder, default parameters.
N = 100;  ξ = 0.1;  ρ = 0.1;  β = 1.0;  δt = Δt
case = "../data/results/case-2_crit-spd/"
pop = "N-$(N)/xi-$(round( ξ, digits=6 ))/rho-$(round( ρ, digits=6 ))_beta-$(round( β, digits=6))/"

if ~isdir( case*pop )
    mkpath( case*pop )
end

In [ ]:
# Parameter conditions for simulation.
if loadsteps
    μlist = vcat( readdlm( case*pop*"mu-list.txt" )... )
    Nμ = length( μlist )
else
    # Draw random parameters for β and τ in the range.
    μmin = -1;  μmax = 1
    Nμ = 51;  Δμ = (μmax - μmin)/(Nμ-1)
    μlist = round.( 10.0.^(μmin:Δμ:μmax), digits=6 )
end;

println( "Running simulation for $(Nμ) unique parameter cases." )

In [ ]:
# Length and time scale.
ϕ = 0.25
scale = Scale( 1.0, ξ )

# Adapatable time-step length.
smin = -3;  smax = 0
Ns = Nμ;  Δs = (smax - smin)/(Ns-1)
slist = round.( 10.0.^(smin:Δs:smax)./(scale.T^(ϕ + 1)), digits=6 )  # DIMENSIONAL.
println( "Running for $(Ns) different values of s0." )

# System parameters.
Llist = .√(N./μlist)

# Activity transition variables.
η = 1/500;  γ = 0.1;  τ = 75.0

# Movement variables.
λ = 0.50

# Sensing area parameter.
α = (1/4)π

# Generate parameter variable.
dparams = Params(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=1.0, α=α )
dnondim = Nondim( dparams; scale=scale )

# Create parameter sets.
paramslist = [Params(; ρ=ρ, η=η, β=β, γ=γ, τ=τ, ξ=ξ, λ=λ, ϕ=ϕ, s=s, α=α ) for s ∈ slist]
nondimlist = [Nondim( params; scale=scale ) for params ∈ paramslist];

In [ ]:
# Run simulation under each environment parameter.
T = round( defInt, 500/scale.T );  M = 10
Tload = round( defInt, 500/scale.T )  # If applicable.

# Compute simulation time-step.
Nt = round( defInt, T/δt );  tlist = 1:Nt
nt = round( defInt, 1/(2*δt*scale.T) );  tsave = Set( 1:nt:Nt );

# Frequency of adjacency calculation.
δt̂ = round( defInt, 0.1/δt );

In [ ]:
# Save default set.
if true
    saveparams( case*"default-params.json", dparams )
    savescale( case*pop*"default-scale.json", scale )
end;

# Data folder name.
folderdata = [[case*pop*"mu-$(round( μ, digits=6 ))/spd-$(round( s, digits=6 ))/"
    for s ∈ slist] for μ ∈ μlist];

In [ ]:
# Initialize list and run simulation.
xdatadata = [[[Matrix{defFloat}( undef, length( tsave ) + 1, 3 ) for _ ∈ 1:M] for _ ∈ 1:Ns] for _ ∈ 1:Nμ]
zdatalist = [Matrix{State}( undef, Ns, M ) for _ ∈ 1:Nμ]
@threads for k ∈ 1:Nμ
    # Unpack length of environment.
    L = Llist[k]

    # Run simulation.
    for i ∈ 1:Ns
        nondim = nondimlist[i]
        for m ∈ 1:M
            # If steps are already saved, use as initial state.
            file = loadsteps ? folderdata[k][i]*"steps/state_T-$(Tload)_m-$(m).txt" : nothing

            # Initialize agent states.
            z = initialstate( N, L/scale.L; A=1, file=file )
            ẑ = copystate( z )

            # Initialize adjacency and saved state.
            A = proximity( N, L/scale.L, nondim.r, nondim.α, z.x, z.y, z.θ )
            xdatadata[k][i][m][1,:] = statecomposition( N, ẑ )

            # Run simulation.
            t̂ = 2
            for t ∈ tlist
                # Update the adjacency matrix.
                (t % δt̂) == 0 && (A = proximity( N, L/scale.L, nondim.r, nondim.α, z.x, z.y, z.θ ))

                # Step simulation.
                step!( N, L/scale.L, nondim, z, ẑ; A=A, δt=δt )

                # Save state if in appropriate subset.
                t ∈ tsave && (xdatadata[k][i][m][t̂,:] = statecomposition( N, ẑ ); t̂ += 1)

                # Swap contents.
                tmp = z;  z = ẑ;  ẑ = tmp
            end

            # Save last simulation state.
            zdatalist[k][i,m] = z
        end
    end
end

In [ ]:
# Compute determinism metric and related statistics.
Rdatalist = [Matrix{RecurrenceMap}( undef, Ns, M ) for _ ∈ 1:Nμ]
ςdatalist = [Matrix{defFloat}( undef, Ns, M ) for _ ∈ 1:Nμ]
for k ∈ 1:Nμ
    for i ∈ 1:Ns
        for m ∈ 1:M
            Rdatalist[k][i,m] = recurrence( xdatadata[k][i][m]; δx=1/100 )
            ςdatalist[k][i,m] = determinism( Rdatalist[k][i,m]; ℓ0=15 )
        end
    end
end

# Determinism statistics.
ς̄data = [vcat( mean( ςdata, dims=2 )... ) for ςdata ∈ ςdatalist];

In [ ]:
# Plot the determinism in each case for visual inspection.
plt = plot( size=(400,200), xformatter=:plain, dpi=600 )

for (k, ς̄list) ∈ enumerate( ς̄data[1:10:Nμ] )
    plot!( plt, slist*scale.T^(ϕ + 1), ς̄list; lw=2, marker=:circ, label="" )
        # label=latexstring( "τ=$(τlist[k]), β=$(βlist[k])" ) )
end

plot!( plt; xlims=10.0.^[-3,0], xscale=:log10 )
plot!( plt; ylims=(0,1) )

plot!( plt; xlabel="effective temperature, "*L"s_0", ylabel="determinism", legend=:outerright )

In [ ]:
k = 2;  i = 2
println( (ς̄data[k][i], μlist[k], slist[i], criticals( paramslist[i], N, μlist[k] )) )

# Plot the time-series for a single parameter case and rho.
plt = plot( size=(400,200), xformatter=:plain, margin=10pt, dpi=600 )

for m ∈ 1:M
    alist = xdatadata[k][i][m][:,1]
    alpha = m==M ? 1 : 1/10
    plot!( plt, δt*(0:nt:Nt), alist; color=:black, alpha=alpha, lw=2, label="" )
end

plot!( plt; xlims=(0,T), xlabel="time, "*L"t" )
plot!( plt; ylims=(0,1), ylabel="proportion of\nants active, "*L"a" )

In [ ]:
for (k, folderlist) ∈ enumerate( folderdata )
    for (i, folder) ∈ enumerate( folderlist )
        if !isdir( folder*"steps/" )
            mkpath( folder*"steps/" )
        end

        if true
            # Save variables of interest.
            xdata = xdatadata[k][i]
            params = paramslist[i]

            # Save macroscopic measurements.
            writedlm( folder*"activity_T-$(round( defInt, T )).txt", [xlist[:,1] for xlist ∈ xdata] )
            writedlm( folder*"inactivity_T-$(round( defInt, T )).txt", [xlist[:,2] for xlist ∈ xdata] )
            writedlm( folder*"refractory_T-$(round( defInt, T )).txt", [xlist[:,3] for xlist ∈ xdata] )
            writedlm( folder*"determinism_T-$(round( defInt, T )).txt", ςdatalist[k][i,:] )

            # Save dimensional parameters and scale.
            saveparams( folder*"params.json", params )
            savescale( folder*"scale.json", scale )
        end
    end
end

if savesteps
    println( "Saving final step of simulation for future initial conditions." )

    for k ∈ 1:Nμ for i ∈ 1:Ns for m ∈ 1:M
        folder = folderdata[k][i]
        savestate( folder*"steps/state_T-$(T)_m-$(m).txt", zdatalist[k][i,m] )
    end; end; end

    # Compile a cumulative list.
    μ̂list = isfile(  case*pop*"mu-list.txt" ) ? vcat( readdlm(  case*pop*"mu-list.txt" ), μlist ) : μlist

    # Append to existing data set if new data.
    writedlm( case*pop*"mu-list.txt", Set( μ̂list ) )
end